
- 채점한 시트는 덮어씌우는 게 아니라 다운받을 수 있도록 해야 해
- 만든 RAG 체인 코드는 모든 기업의 파일로 실행 가능해야 함
- 사업보고서(=2025_business), 지속가능경영보고서(2025_sustainability), 기업지배구조보고서(2025_governance)는 pdf로 제공
- 사용 모델은 gemini-2.5-flash

오픈 AI API는 임베딩 전용으로, text-embedding-3-small
벡터스토어: Chroma (로컬, Google Drive에 영속 저장)
청크 크기: 800 tokens / 오버랩 100 tokens
BM25 토크나이저: Kiwi (한국어 형태소 분석)


스키마 정의는 다음을 지켜줘
년 = year 예시: 2025
문서 유형 = doc_type   예시: business, governance, sustainability
문서 명 = doc_id 예시: '기업코드'_2025_business
청크 = chunk_id 예시: '기업코드'_2025_business_0142

E(환경) 활동 채점을 위한 임베딩/벡터스토어 구축, RAG 체인, Gemini 채점 로직은 이 노트북에 포함하지 않았습니다 (별도 노트북에서 진행).

## Setup Environment


In [1]:
!pip install pandas pypdf langchain langchain-community kiwipiepy tiktoken

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 8.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.1/385.1 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.6 MB/s eta 0:00:00
  Created wheel for kiwipiepy_model: filename=kiwipiepy_model-0.23.0-py3-none-any.whl size=88067826 sha256=33d049bc1bc1ec01ffe17dce4c7a047827663005111e3eef028db03bbbb53de1
  Stored in directory: /root/.cache/pip/wheels/ec/af/43/b8cbacbd3ffba62b7097221eca55b7a4bfd7bea5d2eaadcbeb
Successfully built kiwipiepy_model
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstall

## PDF 문서 로드 및 청크 분리

제공된 PDF 문서를 로드하고, `Kiwipiepy`를 활용한 사용자 정의 텍스트 분리기로 문서를 청크로 분리합니다. 이때, 요청하신 스키마에 맞춰 각 청크에 대한 메타데이터를 생성합니다.

In [1]:
import os
import glob
import tiktoken
from collections import defaultdict
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from kiwipiepy import Kiwi

# 프로젝트 폴더 내 모든 PDF 파일을 자동으로 탐색 (특정 기업에 국한되지 않고, 모든 기업 파일에 대해 실행 가능)
# 파일명 규칙: '기업코드_년도_문서유형.pdf' (예: 011070_2025_business.pdf)
pdf_dir = '/content/drive/MyDrive/프로젝트'
pdf_files = glob.glob(os.path.join(pdf_dir, '*.pdf'))
print(f"총 {len(pdf_files)}개의 PDF 파일을 찾았습니다.")
for f in pdf_files:
    print(f" - {os.path.basename(f)}")

# Kiwi 형태소 분석기 초기화 (한국어 형태소 단위 토큰화 -> BM25 검색에서 사용)
kiwi = Kiwi()

def kiwi_tokenize(text):
    # Kiwi로 형태소 분석하여 형태소(form) 리스트 반환
    return [token.form for token in kiwi.tokenize(text)]

# 토큰 수 기준 length_function (OpenAI 임베딩 모델과 동일한 tiktoken 인코딩 사용)
tiktoken_encoding = tiktoken.get_encoding("cl100k_base")

def tiktoken_len(text):
    return len(tiktoken_encoding.encode(text))

# 청크 크기 및 오버랩 설정 (토큰 기준)
chunk_size = 800
chunk_overlap = 100

# 토큰 수 기준으로 분리하는 텍스트 스플리터
# (Kiwi 형태소 분석기는 청크 분할이 아니라, 아래에서 BM25 리트리버의 검색용 토크나이저로 사용됩니다)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    length_function=tiktoken_len,
    add_start_index=True
)

all_docs = []

for pdf_file in pdf_files:
    loader = PyPDFLoader(pdf_file)
    documents = loader.load()

    # 파일명에서 정보 추출
    filename = os.path.basename(pdf_file)
    parts = filename.replace('.pdf', '').split('_')

    if len(parts) == 3: # '기업코드_년도_문서유형.pdf' 형식
        company_code, year, doc_type = parts
        doc_id_prefix = f'{company_code}_{year}_{doc_type}'
    else: # 예상치 못한 파일명 형식 처리
        company_code = 'unknown'
        year = 'unknown'
        doc_type = 'unknown'
        doc_id_prefix = filename.replace('.pdf', '')

    # 문서 분리 및 메타데이터 추가 (chunk_id는 문서 전체에서 연속 번호를 부여)
    chunk_counter = 0
    for i, doc in enumerate(documents):
        chunks = text_splitter.split_documents([doc])
        for chunk in chunks:
            chunk.metadata.update({
                "company_code": company_code,
                "year": year,
                "doc_type": doc_type,
                "doc_id": doc_id_prefix,
                "chunk_id": f"{doc_id_prefix}_{chunk_counter:04d}",
                "page_number": doc.metadata.get('page', 'unknown') # 원본 페이지 번호 추가
            })
            all_docs.append(chunk)
            chunk_counter += 1

print(f"총 {len(all_docs)}개의 청크가 생성되었습니다. (청크 크기: 토큰 수 기준 800 / 오버랩 100)")

# 기업 코드별로 청크를 그룹화 (기업별 채점 시 사용)
docs_by_company = defaultdict(list)
for doc in all_docs:
    docs_by_company[doc.metadata['company_code']].append(doc)

company_codes = sorted(docs_by_company.keys())
print(f"\n로드된 기업: {len(company_codes)}개 -> {company_codes}")

# 첫 5개 청크의 메타데이터 확인
for i in range(min(5, len(all_docs))):
    print(f"\nChunk {i+1} metadata: {all_docs[i].metadata}")
    print(f"Chunk {i+1} content (first 100 chars): {all_docs[i].page_content[:100]}...")


ModuleNotFoundError: No module named 'langchain_community'

## 청크 인덱싱 결과 CSV로 저장

방금 생성한 `all_docs`(청크 인덱싱 결과)를 팀 공용 포맷과 유사하게 CSV로 내보냅니다.
컬럼: `chunk_id`, `doc_type`, `page`, `section_title`, `content_type`, `chunk_char_count`, `content_preview`

In [ ]:
import pandas as pd

# section_title / content_type은 현재 청킹 파이프라인에 추출 로직이 없으므로 간단한 규칙으로 채웁니다.
# - content_type: 청크 내 '|' 문자가 일정 수 이상이면 마크다운 표(table)로, 아니면 일반 텍스트(text)로 간주
# - section_title: 별도의 목차/헤딩 파싱을 붙이기 전까지는 빈 값으로 둡니다.
CONTENT_PREVIEW_LEN = 100

def infer_content_type(text):
    return "table" if text.count("|") >= 4 else "text"

def make_preview(text, length=CONTENT_PREVIEW_LEN):
    single_line = " ".join(text.split())  # 줄바꿈/중복 공백 정리 (CSV 가독성 위해)
    if len(single_line) > length:
        return single_line[:length] + "..."
    return single_line

index_records = []
for doc in all_docs:
    content = doc.page_content
    index_records.append({
        "chunk_id": doc.metadata.get("chunk_id"),
        "doc_type": doc.metadata.get("doc_type"),
        "page": doc.metadata.get("page_number"),
        "section_title": doc.metadata.get("section_title", ""),
        "content_type": infer_content_type(content),
        "chunk_char_count": len(content),
        "content_preview": make_preview(content),
    })

index_df = pd.DataFrame(
    index_records,
    columns=["chunk_id", "doc_type", "page", "section_title",
             "content_type", "chunk_char_count", "content_preview"]
)

# 팀원 포맷과 동일하게 UTF-8 BOM(utf-8-sig)으로 저장 (엑셀에서 한글 깨짐 방지)
index_csv_path = '/content/drive/MyDrive/chunk_index_export.csv'
index_df.to_csv(index_csv_path, index=False, encoding='utf-8-sig')

print(f"청크 인덱싱 결과 CSV가 '{index_csv_path}'에 저장되었습니다. (총 {len(index_df)}행)")
display(index_df.head())
